# ĐỒ ÁN MÔN BIG DATA: PHÂN TÍCH HÀNH VI KHÁCH HÀNG E-COMMERCE
## NOTEBOOK 02: HỌC MÁY PHÂN KHÚC KHÁCH HÀNG (K-MEANS & RFM) VÀ KHAI PHÁ GIỎ HÀNG (APRIORI)
---
- **Học phần:** Dữ liệu Lớn (Big Data Analytics)
- **Thuật toán áp dụng:** K-Means Clustering, Elbow Method, Silhouette Score, PCA, Apriori Algorithm
- **Mục tiêu:** Định danh 4 Chân dung Khách hàng và Khám phá Luật kết hợp Bán chéo (Cross-selling).

### 1. Nạp Dữ liệu Đặc trưng RFM cấp User từ BigQuery

In [2]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

ROOT_DIR = Path.cwd().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.bq_client import BigQueryService
from src.data_processor import CustomerDataProcessor
from src.clustering import CustomerClusterModel
from src.market_basket import MarketBasketAnalyzer

bq = BigQueryService()
query_rfm = bq.read_sql_file("03_rfm_features.sql")
df_rfm = bq.query_to_dataframe(query_rfm, cache_name="rfm_features")

print(f"✓ Nạp thành công {len(df_rfm):,} hồ sơ người dùng từ BigQuery!")
display(df_rfm.head(5))

ℹ️ BigQuery client offline or credentials not set (Your default credentials were not found. To set up Application Default Credentials, see https://cloud.google.com/docs/authentication/external/set-up-adc for more information.). Using smart cache/sample mode.
⚡ Loading cached results for [rfm_features] (rfm_features.parquet)
✓ Nạp thành công 4,500 hồ sơ người dùng từ BigQuery!


user_pseudo_id,recency_days,total_sessions,monetary_value,total_engagement_sec,total_pageviews,items_viewed_count,items_added_to_cart_count,cart_to_view_ratio,purchase_count,primary_device,primary_channel
user_000000.75682867,92,2,0.00,86,3,2,0,0.000,0,mobile,Organic Search
user_000001.66755036,8,2,12.64,382,18,22,5,0.227,1,desktop,Referral
user_000002.66882282,41,4,0.00,383,27,20,2,0.100,0,desktop,Organic Search
user_000003.31081788,156,1,0.00,89,2,1,0,0.000,0,desktop,Organic Search
user_000004.23315092,7,5,0.00,351,29,25,5,0.200,0,mobile,Direct


### 2. Tiền xử lý Dữ liệu (IQR Clipping $\to$ Log1p Transform $\to$ StandardScaler)

In [4]:
processor = CustomerDataProcessor()
X_scaled, df_trans, df_clean = processor.fit_transform(df_rfm)

print("✓ Ma trận đặc trưng sau chuẩn hóa (StandardScaler):")
print("  • Mean:", np.mean(X_scaled, axis=0).round(4))
print("  • Std:", np.std(X_scaled, axis=0).round(4))
print("  • Shape:", X_scaled.shape)

✓ Preprocessing Validation Check Passed:
  • Shape preserved: (4500, 9)
  • Mean range across features: [-0.0000, 0.0000] (All ≈ 0.0)
  • Std range across features:  [1.0000, 1.0000] (All ≈ 1.0)
  • Zero NaN or Inf values detected.
✓ Ma trận đặc trưng sau chuẩn hóa (StandardScaler):
  • Mean: [ 0.  0. -0.  0. -0.  0. -0. -0.  0.]
  • Std: [1. 1. 1. 1. 1. 1. 1. 1. 1.]
  • Shape: (4500, 9)


### 3. Đánh giá Toán học Tìm $K$ Tối ưu (Phương pháp Elbow & Silhouette Score)

In [6]:
cluster_model = CustomerClusterModel(random_state=42)
eval_res = cluster_model.evaluate_k_range(X_scaled, k_min=2, k_max=8)

# Vẽ biểu đồ Elbow & Silhouette song song
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=eval_res["k_values"],
    y=eval_res["inertias"],
    mode="lines+markers",
    name="Inertia (Elbow Method)",
    line=dict(color="#3B82F6", width=3)
))
fig.add_trace(go.Scatter(
    x=eval_res["k_values"],
    y=eval_res["silhouette_scores"],
    mode="lines+markers",
    name="Silhouette Score",
    yaxis="y2",
    line=dict(color="#10B981", width=3)
))
fig.update_layout(
    title="Đánh giá Tìm số Cụm Tối ưu K (Elbow & Silhouette Curve)",
    xaxis=dict(title="Số lượng Cụm (K)"),
    yaxis=dict(title="Inertia (WCSS)"),
    yaxis2=dict(title="Silhouette Score", overlaying="y", side="right"),
    template="plotly_dark"
)
fig.show()

📊 Evaluating K-Means across range K=2..8...
  • K=2: Inertia =   17327.77 | Silhouette Score = 0.5230
  • K=3: Inertia =   10818.51 | Silhouette Score = 0.4750
  • K=4: Inertia =    8610.25 | Silhouette Score = 0.4797
  • K=5: Inertia =    6729.28 | Silhouette Score = 0.4649
  • K=6: Inertia =    6204.87 | Silhouette Score = 0.3990
  • K=7: Inertia =    5806.98 | Silhouette Score = 0.3805
  • K=8: Inertia =    5424.16 | Silhouette Score = 0.3219


### 4. Huấn luyện K-Means ($K=4$) & Định danh 4 Chân dung Khách hàng (Personas)

In [8]:
df_segmented, cluster_summary = cluster_model.fit_predict(X_scaled, df_clean, n_clusters=4)
df_enriched, df_persona_table = cluster_model.profile_personas(df_segmented)

print("=== BẢNG TỔNG HỢP 4 CHÂN DUNG KHÁCH HÀNG (PERSONAS TABLE) ===")
display(df_persona_table)


✓ K-Means Clustering Validation Passed:
  • Overall Silhouette Score: 0.4797
  • PCA 2D Explained Variance: 82.30%
  • PCA 3D Explained Variance: 91.10%
  • Cluster Distribution:
    - Cluster 0: 1,204 users (26.76%)
    - Cluster 1: 1,593 users (35.4%)
    - Cluster 2: 1,329 users (29.53%)
    - Cluster 3: 374 users (8.31%)
=== BẢNG TỔNG HỢP 4 CHÂN DUNG KHÁCH HÀNG (PERSONAS TABLE) ===


Cluster ID,Chân dung Khách hàng (Persona),Số lượng Users,Recency TB (ngày),Sessions TB,Doanh thu TB ($),Thời gian Tương tác TB (s),Số SP Xem TB,Số SP Thêm Giỏ TB,Tỷ lệ Giỏ/Xem,Đơn hàng TB,Hành vi Nổi bật
0,Potential Loyalists,"1,204 (26.76%)",18.1,4.8,105.14,568.3,21.6,4.5,0.234,1.40,"Tương tác đều đặn, xem nhiều sản phẩm, chi tiêu ở mức khá"
1,At-Risk / Inactive Customers,"1,593 (35.4%)",115.2,1.5,1.62,62.8,2.5,0.2,0.107,0.06,"Đã lâu không quay lại website, tần suất và giá trị rất thấp"
2,Cart Abandoners / Window Shoppers,"1,329 (29.53%)",22.7,3.6,0.03,448.1,18.6,4.1,0.251,0.02,Tỷ lệ thêm vào giỏ cao nhưng tỷ lệ hoàn tất đơn hàng rất thấp
3,Loyal Champions (VIPs),374 (8.31%),7.7,14.9,380.60,1463.0,36.8,12.3,0.385,6.03,"Chi tiêu lớn nhất, tương tác lâu, mua sắm định kỳ thường xuyên"


#### Trực quan hóa Không gian Cụm 3D với PCA

In [10]:
fig_3d = px.scatter_3d(
    df_enriched.sample(min(500, len(df_enriched)), random_state=42),
    x="pca_3d_x",
    y="pca_3d_y",
    z="pca_3d_z",
    color="persona_name",
    hover_name="user_pseudo_id",
    title="Không gian Phân cụm Khách hàng 3D (PCA Projection)"
)
fig_3d.update_layout(template="plotly_dark")
fig_3d.show()

### 5. Khai phá Luật Kết hợp Giỏ hàng (Market Basket Analysis - Apriori)

In [12]:
query_market = bq.read_sql_file("04_market_basket.sql")
df_market = bq.query_to_dataframe(query_market, cache_name="market_basket")

mba = MarketBasketAnalyzer(min_support=0.04, min_confidence=0.25, min_lift=1.0)
basket_mat, n_valid = mba.build_basket_matrix(df_market)
freq_sets, rules = mba.run_apriori_and_rules(basket_mat)
df_recs = mba.get_cross_selling_recommendations(top_n=5)

print("=== BẢNG GỢI Ý SẢN PHẨM BÁN CHÉO (CROSS-SELLING RECOMMENDATIONS) ===")
display(df_recs)

⚡ Loading cached results for [market_basket] (market_basket.parquet)
✓ One-Hot Basket Matrix constructed:
  • Total Multi-Item Orders (Rows): 560
  • Distinct Active Products (Cols): 8
✓ Apriori & Association Mining Completed:
  • Frequent Itemsets found: 12
  • High-Lift Association Rules (Lift > 1.0): 8
=== BẢNG GỢI Ý SẢN PHẨM BÁN CHÉO (CROSS-SELLING RECOMMENDATIONS) ===


Sản phẩm Đã mua (A),Sản phẩm Gợi ý Mua kèm (B),Độ phổ biến (Support),Xác suất mua kèm (Confidence),Hệ số Tương quan (Lift),Ý nghĩa Kinh doanh
Google Leatherette Coaster Set,Google Hard Cover Journal,23.0%,78.2%,2.516x,Khách mua [Google Leatherette Coaster Set] có xác suất 78.2% sẽ mua kèm [Google Hard Cover Journal] (Cao gấp 2.516x so với mua ngẫu nhiên).
Google Hard Cover Journal,Google Leatherette Coaster Set,23.0%,74.1%,2.516x,Khách mua [Google Hard Cover Journal] có xác suất 74.1% sẽ mua kèm [Google Leatherette Coaster Set] (Cao gấp 2.516x so với mua ngẫu nhiên).
Google Unisex Eco Tee Black,Google Twill Cap,48.8%,85.6%,1.493x,Khách mua [Google Unisex Eco Tee Black] có xác suất 85.6% sẽ mua kèm [Google Twill Cap] (Cao gấp 1.493x so với mua ngẫu nhiên).
Google Twill Cap,Google Unisex Eco Tee Black,48.8%,85.0%,1.493x,Khách mua [Google Twill Cap] có xác suất 85.0% sẽ mua kèm [Google Unisex Eco Tee Black] (Cao gấp 1.493x so với mua ngẫu nhiên).
Google Thermal Bottle 20oz,Google Twill Cap,27.0%,77.0%,1.344x,Khách mua [Google Thermal Bottle 20oz] có xác suất 77.0% sẽ mua kèm [Google Twill Cap] (Cao gấp 1.344x so với mua ngẫu nhiên).
